# RevIN Ablation — Real LSTM vs Quaternion, normalization de-confounded

The standard comparison gives quaternion models per-window **RevIN** but real LSTMs only a **static Z-score** (training-set stats). In price-mode that puts test prices out-of-distribution for the real baseline by construction, so any quaternion advantage confounds *architecture* with *normalization*.

This notebook runs `real_lstm_revin` / `real_lstm_attention_revin` (same backbones, same RevIN protocol as the quaternion models) head-to-head with the Z-scored baselines and param-matched quaternion models, on **daily and 4-hourly BTC, price and return mode** (4 experiments, 7 variants × 3 seeds each).

**Read the result like this:** if `real_lstm_revin` ≈ quaternion corr, the previously seen gap was normalization, not architecture. Any remaining gap is the architecture effect.

## 1. Setup

In [ ]:
import os
if not os.path.exists('/content/thesis'):
    !git clone https://github.com/BerkayClik/thesis.git /content/thesis
%cd /content/thesis
!git pull

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Copy the BTC daily + 4h cache CSVs from Drive (recursive search).
import os, shutil, glob
os.makedirs('data/cache', exist_ok=True)
DRIVE_DATA_DIR = '/content/drive/MyDrive/thesis_data'
copied = 0
for pat in ['lunarcrush_btc_day*.csv', 'lunarcrush_btc_4hour*.csv']:
    for src in glob.glob(os.path.join(DRIVE_DATA_DIR, '**', pat), recursive=True):
        shutil.copy(src, os.path.join('data/cache', os.path.basename(src))); copied += 1
        print('copied', os.path.basename(src))
if copied == 0:
    print(f'No BTC caches found under {DRIVE_DATA_DIR} (searched recursively).')

In [ ]:
!pip install -q yfinance scipy seaborn uv
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
os.environ['PYTHONPATH'] = '/content/thesis'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [ ]:
# Build the isolated vectorbt backtest env once (needs numpy<2)
!uv venv --python 3.11 .venv-backtest
!uv pip install --python .venv-backtest -r requirements-backtest.txt
!.venv-backtest/bin/python scripts/backtest_env_smoke.py

## 2. Run the 4 ablation experiments (7 variants × 3 seeds each)

> **✅ Resumable / disconnect-proof** — same checkpoint scheme as the other notebooks: results restore from a fixed Drive folder, finished experiments are skipped, and each experiment saves to Drive the moment it completes. Re-run this cell after a disconnect.

In [ ]:
EXP = [
    ('daily',   'btc_ohlc',        'daily_revin_ablation_3seed',   'daily_revin_ablation_ohlc'),          # 1 daily price
    ('daily',   'btc_ohlc_return', 'daily_revin_ablation_3seed',   'daily_revin_ablation_ohlc_return'),   # 2 daily return
    ('4hourly', 'btc_ohlc',        '4hourly_revin_ablation_3seed', '4hourly_revin_ablation_ohlc'),        # 3 4h price
    ('4hourly', 'btc_ohlc_return', '4hourly_revin_ablation_3seed', '4hourly_revin_ablation_ohlc_return'), # 4 4h return
]

import os, glob, shutil

CKPT_DRIVE = '/content/drive/MyDrive/thesis_results_revin_ablation/checkpoint'
os.makedirs(CKPT_DRIVE, exist_ok=True)

def _result_dir(label):
    return f'experiments/results/{label}'

def _is_done(rdir):
    return any('intermediate' not in f for f in glob.glob(f'{rdir}/*.json'))

# 1) Restore prior results from Drive checkpoint into the local results tree
for _, _, _, label in EXP:
    rdir = _result_dir(label)
    saved = os.path.join(CKPT_DRIVE, label)
    if os.path.isdir(saved) and not os.path.exists(rdir):
        shutil.copytree(saved, rdir, dirs_exist_ok=True)
        print('restored from Drive:', label)

# 2) Run each experiment, skipping ones already done; save to Drive after each
for i, (freq, base, exp, label) in enumerate(EXP, 1):
    rdir = _result_dir(label)
    print(f'\n{"="*70}\nEXPERIMENT {i}/4: {label}\n{"="*70}')
    if _is_done(rdir):
        print('  [skip] already finished ->', rdir)
        continue
    !python experiments/run_experiments.py --base-config configs/data/{freq}/{base}.yaml --experiment-config configs/experiments/{exp}.yaml --results-dir {rdir}
    if _is_done(rdir):
        shutil.copytree(rdir, os.path.join(CKPT_DRIVE, label), dirs_exist_ok=True)
        print('  saved to Drive checkpoint:', label)

## 3. Aggregate tables — the ablation readout

In [ ]:
import json, glob, os
import numpy as np, pandas as pd

RESULT_DIRS = {
    'daily·price':   'experiments/results/daily_revin_ablation_ohlc',
    'daily·return':  'experiments/results/daily_revin_ablation_ohlc_return',
    '4h·price':      'experiments/results/4hourly_revin_ablation_ohlc',
    '4h·return':     'experiments/results/4hourly_revin_ablation_ohlc_return',
}
METRICS = ['mape', 'directional_accuracy', 'sharpe_ratio',
           'directional_accuracy_3class', 'sharpe_ratio_3class']

def latest_json(d):
    js = [f for f in glob.glob(f'{d}/*.json') if 'intermediate' not in f]
    return max(js, key=os.path.getmtime) if js else None

agg_rows, sig_rows = [], []
for exp_name, d in RESULT_DIRS.items():
    jf = latest_json(d)
    if not jf:
        print(f'[skip] {exp_name}: no results'); continue
    res = json.load(open(jf))
    for variant, vdata in res['model_results'].items():
        agg = vdata.get('aggregated', {})
        row = {'experiment': exp_name, 'variant': variant}
        for m in METRICS:
            row[m] = agg.get(m, {}).get('mean', np.nan)
        agg_rows.append(row)
        # signal quality: corr/dir-agree per seed, then mean +/- std
        corrs, das = [], []
        for run in vdata['individual_runs']:
            tm = run['test_metrics']
            if not tm.get('predictions'):
                continue
            pr = np.array(tm['predictions'], float) / np.array(tm['prev_closes'], float) - 1
            tr = np.array(tm['targets'], float) / np.array(tm['prev_closes'], float) - 1
            if pr.std() and tr.std():
                corrs.append(np.corrcoef(pr, tr)[0, 1])
                das.append((np.sign(pr) == np.sign(tr)).mean() * 100)
        if corrs:
            sig_rows.append({'experiment': exp_name, 'variant': variant,
                             'corr_mean': round(np.mean(corrs), 4),
                             'corr_std': round(np.std(corrs), 4),
                             'dir_agree_%': round(np.mean(das), 1)})

agg_df = pd.DataFrame(agg_rows).round(3)
sig_df = pd.DataFrame(sig_rows)
pd.set_option('display.width', 220, 'display.max_rows', 120)
print('=== Metrics (mean over 3 seeds) ===')
print(agg_df.to_string(index=False))
print('\n=== Signal quality (mean ± std over 3 seeds) ===')
print(sig_df.to_string(index=False))
print('\nKey comparison: real_lstm_revin vs real_lstm (normalization effect)')
print('               real_lstm_revin vs quaternion_*_param_matched (architecture effect)')

## 4. Backtest the return-mode experiments (long/short, val-selected dead band, all seeds)

In [ ]:
RETURN_DIRS = [
    'experiments/results/daily_revin_ablation_ohlc_return',
    'experiments/results/4hourly_revin_ablation_ohlc_return',
]
OHLC = {
    'daily_revin_ablation_ohlc_return':   ('data/cache/lunarcrush_btc_day_full.csv', '1d'),
    '4hourly_revin_ablation_ohlc_return': ('data/cache/lunarcrush_btc_4hour_full.csv', '4h'),
}
for d in RETURN_DIRS:
    if not glob.glob(f'{d}/*_seed42_predictions.csv'):
        print(f'[skip] {d}: no seed-42 predictions'); continue
    label = os.path.basename(d)
    ohlc_csv, freq = OHLC[label]
    print(f'\n{"="*70}\nBACKTEST ALL: {label}\n{"="*70}')
    !.venv-backtest/bin/python scripts/backtest_all.py \
        --results-dir "{d}" \
        --ohlc {ohlc_csv} \
        --seeds 42,123,2024 --strategy long_short --threshold auto \
        --exit-mode hold --freq {freq} --fees 0.001 --slippage 0.0005 \
        --fee-grid 0,0.0005,0.001,0.0025

## 5. Save everything to Google Drive

In [ ]:
import shutil
from datetime import datetime
GDRIVE = '/content/drive/MyDrive/thesis_results_revin_ablation'
run_dir = f"{GDRIVE}/{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(run_dir, exist_ok=True)
for exp_name, d in RESULT_DIRS.items():
    if os.path.exists(d):
        shutil.copytree(d, f'{run_dir}/{os.path.basename(d)}', dirs_exist_ok=True)
agg_df.to_csv(f'{run_dir}/metrics_table.csv', index=False)
sig_df.to_csv(f'{run_dir}/signal_quality_table.csv', index=False)
print(f'Saved all results + tables to: {run_dir}')

## Notes

- **What this answers:** is the quaternion advantage (price-mode corr ~0.09–0.13 vs real ~0.04 on daily) architecture or normalization? `real_lstm_revin` uses the identical RevIN protocol as the quaternion models — including skipping price-scale denorm in return mode.
- These are fresh runs, so `*_val_predictions.csv` files exist and `--threshold auto` genuinely tunes the dead band on validation (older result dirs fall back to 0).
- The legacy `test_metrics.sharpe_ratio` is still the toy sign-based Sharpe; the fee-aware numbers live in `backtest_all/`.
